# H0lon worker: распознавание речи в Google Colab

Этот ноутбук превращает бесплатную видеокарту Colab в «удалённый распознаватель» для H0lon.
Лекция распознаётся здесь, а ваш компьютер остаётся свободным. Наружу уходит **только звук**
(сжатый до AAC), назад приходит JSON с сегментами и временем слов.

**Как пользоваться**

1. Включите видеокарту: «Среда выполнения → Сменить тип среды выполнения → T4 GPU».
2. Выполните ячейки сверху вниз («Среда выполнения → Выполнить все»). Весь запуск занимает 3–6 минут;
   при первом запуске ещё скачивается модель (около 3 ГБ).
3. Последняя ячейка с результатом печатает **адрес** и **токен**. Впишите их в `h0lon.toml`
   (раздел `[compute]`), и `h0lon extract` будет отправлять звук сюда.
4. **Не закрывайте вкладку** и не давайте сессии простаивать: пока она открыта, worker работает.
   Бесплатный Colab ограничивает время сессии и доступ к GPU, это нормально.

**Безопасность**

- Токен создаётся заново при каждом запуске и никуда не сохраняется, кроме вывода ячейки. Тот, кто
  знает адрес и токен, может занимать вашу сессию распознаванием, поэтому никому их не показывайте.
- Если вы сохраняете копию ноутбука на Диск или в GitHub, сначала очистите вывод:
  «Правка → Очистить все выводы».
- Ноутбук ставит пакет H0lon из GitHub и официальный бинарный файл `cloudflared` (Cloudflare)
  из его релизов; других загрузок нет.

## 1. Настройки

Менять обычно ничего не нужно. `H0LON_SOURCE` — откуда ставить пакет (можно указать ветку или тег:
`git+https://github.com/Filodot/H0lon@имя-ветки`), `MODEL` — какую модель Whisper подготовить заранее.

In [ ]:
H0LON_SOURCE = "git+https://github.com/Filodot/H0lon"
MODEL = "large-v3"  # large-v3 | large-v3-turbo | medium | small
PORT = 8790

## 2. Проверка видеокарты

Без GPU worker тоже запустится, но будет распознавать в десятки раз медленнее. Если ниже написано
«GPU не найден», включите T4 GPU (см. пункт 1 выше) и начните заново.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(
        subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
            capture_output=True,
            text=True,
        ).stdout
    )
    HAVE_GPU = True
else:
    HAVE_GPU = False
    print(
        "GPU не найден: «Среда выполнения → Сменить тип среды выполнения → T4 GPU», "
        "затем начните заново."
    )

## 3. Установка H0lon

Ставится пакет `h0lon` с группами зависимостей `video` (faster-whisper) и `video-gpu` (библиотеки
CUDA для него). Это занимает 1–3 минуты. Предупреждения pip о конфликтах версий с чужими пакетами
Colab можно игнорировать: worker использует только свои.

In [ ]:
!pip install -q "h0lon[video,video-gpu] @ {H0LON_SOURCE}"

## 4. Токен и вспомогательные функции

Токен — случайная строка; worker отвечает на запросы только с ним (заголовок `Authorization: Bearer`).
Он живёт в памяти этой сессии и печатается только в последней ячейке.

In [ ]:
import os
import re
import secrets
import subprocess
import sys
import time
import urllib.request

TOKEN = secrets.token_urlsafe(32)
LOG_WORKER = "/content/h0lon_worker.log"
LOG_TUNNEL = "/content/cloudflared.log"


def tail(path, lines=15):
    try:
        with open(path, encoding="utf-8", errors="replace") as fh:
            return "".join(fh.readlines()[-lines:])
    except OSError:
        return ""


def http_get(url, timeout=10):
    request = urllib.request.Request(url, headers={"Authorization": "Bearer " + TOKEN})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read().decode("utf-8")


print("Токен создан (он будет показан в последней ячейке).")

## 5. Туннель cloudflared

Colab не принимает входящие соединения, поэтому адрес для вашего компьютера даёт «быстрый туннель»
Cloudflare (`*.trycloudflare.com`): аккаунт не нужен. Скачивается официальный бинарный файл
`cloudflared-linux-amd64` из релизов `cloudflare/cloudflared` на GitHub.

In [ ]:
CLOUDFLARED = "/usr/local/bin/cloudflared"
URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(URL, CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)
if os.path.getsize(CLOUDFLARED) < 10_000_000:
    raise RuntimeError("cloudflared скачался не полностью: повторите ячейку")
print(subprocess.run([CLOUDFLARED, "--version"], capture_output=True, text=True).stdout.strip())

## 6. Модель Whisper

Модель скачивается заранее, чтобы первая лекция не ждала загрузки (large-v3 — около 3 ГБ).

In [ ]:
from faster_whisper.utils import download_model

path = download_model(MODEL)
print("Модель готова:", MODEL)

## 7. Запуск worker и туннеля

Worker слушает только `127.0.0.1` внутри Colab, наружу его выводит туннель. Для CUDA процессу
передаются пути к библиотекам из pip (`nvidia-cublas-cu12`, `nvidia-cudnn-cu12`).

In [ ]:
import glob
import importlib.util

lib_dirs = []
spec = importlib.util.find_spec("nvidia")
roots = list(spec.submodule_search_locations) if spec and spec.submodule_search_locations else []
for root in roots:
    lib_dirs += sorted(glob.glob(os.path.join(root, "*", "lib")))

env = dict(os.environ)
env["H0LON_WORKER_TOKEN"] = TOKEN
env["LD_LIBRARY_PATH"] = ":".join([*lib_dirs, env.get("LD_LIBRARY_PATH", "")]).strip(":")

command = [sys.executable, "-m", "h0lon.worker", "--host", "127.0.0.1", "--port", str(PORT)]
with open(LOG_WORKER, "w") as log:
    worker = subprocess.Popen(
        [*command, "--model", MODEL], env=env, stdout=log, stderr=subprocess.STDOUT
    )

for _ in range(60):
    if worker.poll() is not None:
        raise RuntimeError("worker не запустился:\n" + tail(LOG_WORKER, 30))
    try:
        http_get(f"http://127.0.0.1:{PORT}/health", timeout=2)
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("worker не ответил за минуту:\n" + tail(LOG_WORKER, 30))
print("worker запущен:", http_get(f"http://127.0.0.1:{PORT}/health"))

with open(LOG_TUNNEL, "w") as log:
    tunnel = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=log,
        stderr=subprocess.STDOUT,
    )
PUBLIC_URL = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tail(LOG_TUNNEL, 200))
    if found:
        PUBLIC_URL = found.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError("cloudflared завершился:\n" + tail(LOG_TUNNEL, 30))
    time.sleep(1)
if PUBLIC_URL is None:
    raise RuntimeError("cloudflared не выдал адрес за минуту:\n" + tail(LOG_TUNNEL, 30))
print("Туннель открыт.")

## 8. Адрес и токен для h0lon.toml

Скопируйте блок ниже в `h0lon.toml` на своём компьютере (замените раздел `[compute]` или добавьте
эти ключи в него). Адрес начинает отвечать через несколько секунд после создания туннеля, ячейка
сама это проверяет. **Этот вывод содержит токен: не публикуйте его.**

In [ ]:
for _attempt in range(30):
    try:
        print("Проверка через туннель:", http_get(PUBLIC_URL + "/health", timeout=10))
        break
    except OSError as exc:
        last = exc
        time.sleep(2)
else:
    print("Адрес пока не отвечает (", last, "): подождите минуту и выполните ячейку ещё раз.")

print()
print("Впишите в h0lon.toml:")
print()
print("[compute]")
print('asr = "colab"')
print(f'colab_url = "{PUBLIC_URL}"')
print(f'colab_token = "{TOKEN}"')

## 9. Пока идёт работа

Эта ячейка раз в полминуты показывает, занят ли worker, и не даёт сессии простаивать. Остановить её
можно кнопкой «Прервать выполнение»; worker при этом продолжит работать. Если распознавание не
идёт, а сессия закрывается по простою, оставьте ячейку выполняться.

In [ ]:
import json

try:
    while True:
        try:
            state = json.loads(http_get(f"http://127.0.0.1:{PORT}/health"))
            print(
                time.strftime("%H:%M:%S"),
                "занят" if state["busy"] else "свободен",
                f"(в работе {state['running']}, в очереди {state['queued']})",
            )
        except OSError as exc:
            print("worker не отвечает:", exc)
        time.sleep(30)
except KeyboardInterrupt:
    print("Наблюдение остановлено, worker продолжает работать.")

## 10. Остановка

Когда лекции распознаны, остановите worker и туннель (или просто закройте сессию:
«Среда выполнения → Отключиться и удалить среду выполнения»).

In [ ]:
for process in (tunnel, worker):
    process.terminate()
print("Worker и туннель остановлены.")